# Python preprocessing pipeline pro vektorovou říční síť

Tento notebook staví hlavní část projektu jako čistý Python/GIS preprocessing workflow pro segmentovanou vektorovou říční síť. Cílem není automatizovat konkrétní desktopový GIS, ale připravit korektní dataset, který lze následně symbolizovat, vložit do layoutu a exportovat v libovolném GIS software.

Pipeline je inspirovaná původním Japan MLIT W05 river-network case. Ten zůstává reálným vstupním profilem, protože určuje field names, regionální členění a některé datové zvláštnosti. Samotný preprocessing je ale formulovaný obecněji: načtení dat, validace schématu, kontrola CRS, čištění liniových segmentů, zachování geometrie, regionální obohacení, výpočet metrik a export vrstvy připravené pro kartografickou vizualizaci.


## 1. Scope notebooku

Hlavní výstup notebooku je vrstva `river_network_prepared`, typicky ve formátu GeoPackage. Vrstva obsahuje původní liniovou geometrii, ověřené CRS metadata, regionální atributy a odvozené metriky použitelné pro symbolizaci.

Do hlavní části patří:

- načtení a discovery vstupních vektorových dat,
- kontrola atributového schématu,
- validace a harmonizace CRS,
- čištění a filtrace říčních segmentů,
- výpočet délek a síťových metrik,
- příprava exportního datasetu pro GIS symbolizaci.

Do hlavní části nepatří software-specific layout automation. ArcGIS Pro, QGIS nebo jiný GIS je zde až navazující prostředí, které převezme připravený dataset.


## 2. Import knihoven a nastavení cest

Notebook používá běžný Python geospatial stack: `geopandas`, `pandas`, `shapely` a `pyproj`. Výpočet nejdelší vzdálenosti od pramenných uzlů využívá projektový modul `river_pipeline.network`, který je čistý Python a není navázaný na žádné desktopové GIS API.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from river_pipeline.config import JAPAN_W05_PROFILE
from river_pipeline.discovery import (
    group_folders,
    keyword_score,
    safe_name,
    vector_files,
)
from river_pipeline.network import compute_source_distances

try:
    import geopandas as gpd
    import pandas as pd
    from pyproj import CRS
except ImportError as exc:
    raise RuntimeError(
        "Notebook vyžaduje běžný geospatial Python stack. "
        "Nainstaluj geopandas, pandas, shapely a pyproj v aktivním environmentu."
    ) from exc


def read_vector(path, **kwargs):
    """Read vector data while tolerating isolated invalid source geometries."""
    try:
        return gpd.read_file(path, engine="pyogrio", on_invalid="warn", **kwargs)
    except TypeError:
        return gpd.read_file(path, **kwargs)


## 3. Konfigurační profil datasetu

Profil drží field mapping a dataset-specific pravidla. Pro tento projekt používáme `JAPAN_W05_PROFILE`, protože vstupní data jsou Japan MLIT W05 shapefiles. Kritické field names neměníme, pouze je čteme z konfigurace, aby zůstalo jasné, které názvy jsou vlastností konkrétního zdroje a které části pipeline jsou obecné.


In [ ]:
profile = JAPAN_W05_PROFILE
schema = profile.schema
region = profile.region

RAW_ROOT = PROJECT_ROOT / "data"
REGIONS_ROOT = PROJECT_ROOT / "regions"
PREFECTURE_BOUNDARY_PATH = REGIONS_ROOT / "jpn_admbnda_adm1_2019.shp"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "python_preprocessing"
OUTPUT_LAYER = "river_network_prepared"
NATIONAL_OUTPUT_DIR = OUTPUT_ROOT / "national"
NATIONAL_OUTPUT_PATH = NATIONAL_OUTPUT_DIR / "national_river_network_prepared.gpkg"

PREFECTURE_CODE_FIELD_CANDIDATES = ("PREF_CODE", "ADM1_PCODE", "ADM1_CODE", "JCODE", "KEN_CD")
PREFECTURE_NAME_FIELD_CANDIDATES = ("PREF_NAME", "ADM1_EN", "ADM1_JA", "ADM1_NAME", "NAME_1", "KEN")

# Pokud je vstupní shapefile bez .prj, používáme jen ověřený metadata fallback.
# U W05 odpovídají extent hodnoty geografickým souřadnicím nad Japonskem.
FALLBACK_CRS = profile.fallback_crs

# None znamená zpracovat všechny nalezené processing groups.
# Pro rychlé testování lze nastavit například GROUP_LIMIT = 3.
GROUP_LIMIT = None

print(profile.name)
print("Input root:", RAW_ROOT)
print("Prefecture boundaries:", PREFECTURE_BOUNDARY_PATH)
print("Output root:", OUTPUT_ROOT)


## 4. Discovery vstupních datasetů

Vstupy jsou organizované po processing groups. V tomto konkrétním případě jedna složka odpovídá prefektuře, ale pipeline s ní pracuje obecně jako s jednou skupinou dat. Toky a uzly vybíráme podle názvu souboru a následně jejich schéma ověřujeme až po načtení.


In [ ]:
def choose_by_keywords(files, keywords):
    ranked = sorted(((keyword_score(path, keywords), path) for path in files), reverse=True)
    return ranked[0][1] if ranked and ranked[0][0] > 0 else None


def discover_inputs(raw_root, profile, group_limit=None):
    discovered = []
    for group in group_folders(raw_root)[:group_limit]:
        files = vector_files(group.path, extensions=(".shp", ".gpkg", ".geojson"))
        stream_path = choose_by_keywords(files, profile.stream_keywords)
        node_path = choose_by_keywords(files, profile.node_keywords)
        discovered.append({
            "group": group,
            "stream_path": stream_path,
            "node_path": node_path,
            "file_count": len(files),
        })
    return discovered

inputs = discover_inputs(RAW_ROOT, profile, GROUP_LIMIT)
summary = pd.DataFrame([
    {
        "group": item["group"].name,
        "stream_path": str(item["stream_path"]) if item["stream_path"] else None,
        "node_path": str(item["node_path"]) if item["node_path"] else None,
        "file_count": item["file_count"],
    }
    for item in inputs
])
summary.head()


## 5. Schema validation

Schéma kontrolujeme před jakýmkoliv výpočtem. Povinná jsou pole pro směr toku a pro orientovanou hranu sítě: start node, end node a později délka segmentu. Délku můžeme dopočítat z geometrie, proto nemusí být ve vstupu přítomná. Ostatní atributy, například název nebo kód řeky, jsou užitečné, ale nejsou execution-critical.


In [ ]:
def validate_stream_schema(gdf, schema):
    required_fields = [
        schema.flow_status_field,
        schema.start_node_field,
        schema.end_node_field,
    ]
    required_fields = [field for field in required_fields if field]
    missing = [field for field in required_fields if field not in gdf.columns]

    optional_fields = [
        schema.river_name_field,
        schema.river_code_field,
        schema.length_field,
    ]
    optional_fields = [field for field in optional_fields if field]
    present_optional = [field for field in optional_fields if field in gdf.columns]

    geometry_types = sorted(gdf.geometry.geom_type.dropna().unique().tolist())
    return {
        "feature_count": len(gdf),
        "geometry_types": geometry_types,
        "missing_required_fields": missing,
        "present_optional_fields": present_optional,
        "columns": list(gdf.columns),
    }

sample_item = next(item for item in inputs if item["stream_path"] is not None)
sample_gdf = read_vector(sample_item["stream_path"])
schema_report = validate_stream_schema(sample_gdf, schema)
schema_report


## 6. CRS validation a harmonizace

CRS řešíme ve dvou různých rovinách. Pokud dataset nemá CRS metadata, můžeme mu po kontrole extentu přiřadit fallback CRS z profilu. To pouze doplní metadata, souřadnice se tím nepřepočítávají. Pro délkové metriky následně pracujeme s dočasnou projektovanou kopií, aby délky nebyly počítané ve stupních.

Geometrie exportního datasetu zůstává v harmonizovaném vstupním CRS. Odvozené délkové fieldy se počítají v metrickém CRS a ukládají se jako atributy.


In [ ]:
def harmonize_crs(gdf, fallback_crs=None):
    if gdf.crs is not None:
        return gdf, "source_crs"
    if fallback_crs is None:
        raise ValueError("Dataset nemá CRS metadata a profil neposkytuje fallback_crs.")
    return gdf.set_crs(fallback_crs, allow_override=False), "fallback_crs"


def estimate_metric_crs(gdf):
    try:
        estimated = gdf.estimate_utm_crs()
        if estimated is not None:
            return estimated
    except Exception:
        pass
    return CRS.from_epsg(3857)

sample_harmonized, crs_source = harmonize_crs(sample_gdf, FALLBACK_CRS)
metric_crs = estimate_metric_crs(sample_harmonized)

print("CRS source:", crs_source)
print("Export CRS:", sample_harmonized.crs)
print("Metric CRS for derived lengths:", metric_crs)
print("Bounds:", sample_harmonized.total_bounds)


## 7. Cleaning a filtering říčních segmentů

Čištění je konzervativní. Zahazujeme prázdné nebo při načtení invalidní geometrie, neliniové prvky a segmenty bez známého směru toku. U W05 je známý tok reprezentovaný hodnotou `W05_006 == 1`, ale samotná funkce čte pravidla z profilu. Geometrii segmentů neopravujeme ani negeneralizujeme, protože cílem je preprocessing pro symbolizaci, ne kartografická generalizace.


In [ ]:
LINE_GEOMETRY_TYPES = {"LineString", "MultiLineString"}


def known_flow_mask(gdf, field, known_values):
    if field is None or field not in gdf.columns:
        return pd.Series(True, index=gdf.index)
    normalized = gdf[field].astype(str).str.strip().str.upper()
    expected = {str(value).strip().upper() for value in known_values}
    return normalized.isin(expected)


def clean_segments(gdf, schema):
    original_count = len(gdf)
    non_empty = gdf.geometry.notna() & ~gdf.geometry.is_empty
    line_like = gdf.geometry.geom_type.isin(LINE_GEOMETRY_TYPES)
    flow_known = known_flow_mask(gdf, schema.flow_status_field, schema.known_flow_values)

    cleaned = gdf.loc[non_empty & line_like & flow_known].copy()
    report = {
        "input_features": original_count,
        "removed_empty_geometry": int((~non_empty).sum()),
        "removed_non_line_geometry": int((non_empty & ~line_like).sum()),
        "removed_unknown_flow": int((non_empty & line_like & ~flow_known).sum()),
        "output_features": len(cleaned),
    }
    return cleaned, report

sample_cleaned, cleaning_report = clean_segments(sample_harmonized, schema)
cleaning_report


## 8. Geometry preservation checkpoint

Po cleaningu ověříme, že pipeline pouze vybírala segmenty a přidávala atributy. Nemá měnit souřadnice ani tvar linií. Prakticky kontrolujeme CRS, typy geometrií a základní bounding box před a po filtraci.


In [ ]:
def geometry_preservation_report(before, after):
    return {
        "crs_preserved": before.crs == after.crs,
        "input_bounds": before.total_bounds.tolist(),
        "output_bounds": after.total_bounds.tolist(),
        "output_geometry_types": sorted(after.geometry.geom_type.dropna().unique().tolist()),
    }

geometry_preservation_report(sample_harmonized, sample_cleaned)


## 9. Derived metrics pro vizualizaci

Délku segmentu počítáme v metrickém CRS na dočasné kopii. Následně používáme start/end node fieldy pro výpočet vzdálenosti od pramenných částí sítě. Výsledek doplňujeme několika jednoduchými kartografickými atributy: kvantilovou třídou vzdálenosti a doporučenou relativní šířkou linie. Tyto atributy nejsou symbologie sama, pouze připravená data pro pozdější stylování.


In [ ]:
def add_length_meters(gdf, length_field):
    metric_crs = estimate_metric_crs(gdf)
    projected = gdf.to_crs(metric_crs)
    enriched = gdf.copy()
    enriched[length_field] = projected.geometry.length.astype(float)
    return enriched, metric_crs


def add_source_distance_metrics(gdf, schema):
    required = [schema.start_node_field, schema.end_node_field, schema.length_field]
    missing = [field for field in required if field not in gdf.columns]
    if missing:
        raise ValueError(f"Chybí pole pro network metrics: {missing}")

    metric_df = gdf.reset_index(drop=False).rename(columns={"index": "_row_id"})
    records = metric_df[["_row_id", schema.start_node_field, schema.end_node_field, schema.length_field]].to_dict("records")
    downstream, upstream, stats = compute_source_distances(
        records,
        start_field=schema.start_node_field,
        end_field=schema.end_node_field,
        length_field=schema.length_field,
        id_field="_row_id",
    )

    enriched = gdf.copy()
    enriched[schema.distance_from_source_field] = metric_df["_row_id"].map(downstream).values
    if schema.upstream_length_field:
        enriched[schema.upstream_length_field] = metric_df["_row_id"].map(upstream).values
    return enriched, stats


def add_visualization_fields(gdf, distance_field):
    enriched = gdf.copy()
    distances = pd.to_numeric(enriched[distance_field], errors="coerce")
    valid = distances.dropna()

    if len(valid) >= 4 and valid.nunique() >= 4:
        enriched["DIST_CLASS"] = pd.qcut(distances, q=4, labels=[1, 2, 3, 4], duplicates="drop").astype("Int64")
    else:
        enriched["DIST_CLASS"] = pd.Series(pd.NA, index=enriched.index, dtype="Int64")

    max_distance = valid.max() if len(valid) else 0
    if max_distance > 0:
        enriched["SYMB_WIDTH"] = 0.4 + 2.6 * (distances.fillna(0) / max_distance)
    else:
        enriched["SYMB_WIDTH"] = 0.4
    return enriched

sample_metrics, used_metric_crs = add_length_meters(sample_cleaned, schema.length_field)
sample_metrics, network_stats = add_source_distance_metrics(sample_metrics, schema)
sample_prepared = add_visualization_fields(sample_metrics, schema.distance_from_source_field)

print("Metric CRS used for length:", used_metric_crs)
print(network_stats)
sample_prepared[[schema.length_field, schema.distance_from_source_field, "DIST_CLASS", "SYMB_WIDTH"]].describe()


## 10. Export datasetu připraveného pro GIS symbolizaci

Export zapisuje GeoPackage, protože jde o otevřený, přenositelný formát vhodný pro ArcGIS Pro, QGIS i další GIS software. Před zápisem se říční linie protnou s polygonovou vrstvou prefektur pomocí overlay/intersection, takže segmenty na hranici prefektur se rozdělí a každá výsledná část dostane odpovídající `PREF_CODE` a `PREF_NAME`. Po overlay se znovu přepočítá `LEN_M` pro výsledné geometrie.

Pipeline zachová jednotlivé výstupy po processing groups a zároveň vytvoří jednu národní vrstvu v `outputs/python_preprocessing/national/`.


In [ ]:
def choose_existing_field(gdf, candidates, field_kind):
    for candidate in candidates:
        if candidate in gdf.columns:
            return candidate
    raise ValueError(
        f"Prefecture boundary layer does not contain a usable {field_kind} field. "
        f"Tried: {list(candidates)}. Available fields: {list(gdf.columns)}"
    )


def load_prefecture_boundaries(path, target_crs, region_config):
    if not path.exists():
        raise FileNotFoundError(f"Prefecture boundary layer not found: {path}")

    prefectures = read_vector(path)
    if prefectures.empty:
        raise ValueError(f"Prefecture boundary layer is empty: {path}")
    if prefectures.crs is None:
        raise ValueError(f"Prefecture boundary layer has no CRS metadata: {path}")

    code_field = choose_existing_field(prefectures, PREFECTURE_CODE_FIELD_CANDIDATES, "code")
    name_field = choose_existing_field(prefectures, PREFECTURE_NAME_FIELD_CANDIDATES, "name")

    prefectures = prefectures[[code_field, name_field, "geometry"]].copy()
    prefectures = prefectures[prefectures.geometry.notna() & ~prefectures.geometry.is_empty]
    prefectures = prefectures.rename(columns={
        code_field: region_config.output_code_field,
        name_field: region_config.output_name_field,
    })

    if target_crs is not None and prefectures.crs != target_crs:
        prefectures = prefectures.to_crs(target_crs)
    return prefectures


def assign_region_by_overlay(gdf, prefectures, region_config):
    if region_config is None:
        return gdf

    output_fields = [region_config.output_code_field, region_config.output_name_field]
    source = gdf.drop(columns=output_fields, errors="ignore").copy()
    source["_source_row"] = range(len(source))

    regions_for_overlay = prefectures[output_fields + ["geometry"]].copy()
    if source.crs != regions_for_overlay.crs:
        regions_for_overlay = regions_for_overlay.to_crs(source.crs)

    overlaid = gpd.overlay(source, regions_for_overlay, how="intersection", keep_geom_type=True)

    matched_rows = set(overlaid["_source_row"].dropna().tolist()) if not overlaid.empty else set()
    unmatched = source.loc[~source["_source_row"].isin(matched_rows)].copy()
    for field in output_fields:
        unmatched[field] = pd.NA

    enriched = pd.concat([overlaid, unmatched], ignore_index=True)
    return gpd.GeoDataFrame(enriched.drop(columns=["_source_row"]), geometry="geometry", crs=source.crs)


def process_group(item, profile, output_root):
    schema = profile.schema
    if item["stream_path"] is None:
        return {"group": item["group"].name, "status": "skipped", "reason": "stream dataset not found"}

    gdf = read_vector(item["stream_path"])
    schema_report = validate_stream_schema(gdf, schema)
    if schema_report["missing_required_fields"]:
        return {
            "group": item["group"].name,
            "status": "failed",
            "reason": f"missing required fields: {schema_report['missing_required_fields']}",
        }

    gdf, crs_source = harmonize_crs(gdf, profile.fallback_crs)
    cleaned, cleaning_report = clean_segments(gdf, schema)
    enriched, metric_crs = add_length_meters(cleaned, schema.length_field)
    enriched, network_stats = add_source_distance_metrics(enriched, schema)
    prefectures = load_prefecture_boundaries(PREFECTURE_BOUNDARY_PATH, enriched.crs, profile.region)
    enriched = assign_region_by_overlay(enriched, prefectures, profile.region)
    enriched, metric_crs = add_length_meters(enriched, schema.length_field)
    prepared = add_visualization_fields(enriched, schema.distance_from_source_field)

    group_output = output_root / safe_name(item["group"].name)
    group_output.mkdir(parents=True, exist_ok=True)
    export_path = group_output / f"{safe_name(item['group'].name)}_river_network_prepared.gpkg"
    prepared.to_file(export_path, layer=OUTPUT_LAYER, driver="GPKG")

    return {
        "group": item["group"].name,
        "status": "ok",
        "output": str(export_path),
        "crs_source": crs_source,
        "export_crs": str(prepared.crs),
        "metric_crs": str(metric_crs),
        "input_features": cleaning_report["input_features"],
        "cleaned_features": cleaning_report["output_features"],
        "output_features": len(prepared),
        "removed_unknown_flow": cleaning_report["removed_unknown_flow"],
        "region_assigned_features": int(prepared[profile.region.output_code_field].notna().sum()) if profile.region else None,
        "region_unassigned_features": int(prepared[profile.region.output_code_field].isna().sum()) if profile.region else None,
        "network_segments": network_stats.segment_count,
        "network_nodes": network_stats.node_count,
        "cycle_detected": network_stats.cycle_detected,
        "max_distance_m": network_stats.max_distance,
    }


def export_national_layer(report_df, output_path, layer_name):
    ok_outputs = report_df[report_df["status"].eq("ok")].copy()
    if ok_outputs.empty:
        return {"status": "skipped", "reason": "no successful group outputs"}

    parts = []
    target_crs = None
    for row in ok_outputs.itertuples(index=False):
        part = read_vector(row.output, layer=layer_name)
        part["PROCESSING_GROUP"] = row.group
        if target_crs is None:
            target_crs = part.crs
        elif part.crs != target_crs:
            part = part.to_crs(target_crs)
        parts.append(part)

    national = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs=target_crs)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    national.to_file(output_path, layer=layer_name, driver="GPKG")
    return {"status": "ok", "output": str(output_path), "features": len(national)}


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
reports = [process_group(item, profile, OUTPUT_ROOT) for item in inputs]
report_df = pd.DataFrame(reports)
national_report = export_national_layer(report_df, NATIONAL_OUTPUT_PATH, OUTPUT_LAYER)
report_path = OUTPUT_ROOT / "preprocessing_report.csv"
report_df.to_csv(report_path, index=False)

print("Report:", report_path)
print("National output:", national_report)
report_df.head()


## 11. Kontrola výstupu

Výstupní report je QA stopa preprocessing běhu. Pro každou skupinu ukazuje, zda se dataset podařilo zpracovat, kolik prvků vstoupilo a vystoupilo, jaké CRS bylo použito pro export a jaké metrické CRS bylo použito pro délkové výpočty.


In [ ]:
ok_outputs = report_df[report_df["status"].eq("ok")]
print("Processed groups:", len(ok_outputs), "/", len(report_df))
print("Prepared feature count:", int(ok_outputs["output_features"].sum()) if not ok_outputs.empty else 0)
print("National output:", national_report.get("output"))
print("National feature count:", national_report.get("features", 0))
print("Groups with detected cycles:", int(ok_outputs["cycle_detected"].fillna(False).sum()) if not ok_outputs.empty else 0)

if not ok_outputs.empty:
    first_output = Path(ok_outputs.iloc[0]["output"])
    check = read_vector(first_output, layer=OUTPUT_LAYER)
    print("First output:", first_output)
    print("Fields:", list(check.columns))
    print("CRS:", check.crs)
    print("Features:", len(check))


## 12. Navazující GIS vizualizace a export map

Tímto notebookem končí preprocessing pipeline. Výsledný GeoPackage je připravený pro navazující kartografickou práci:

- symbolizace podle `DIST_CLASS`, `SYMB_WIDTH`, délky segmentu, regionu nebo vzdálenosti od pramene,
- tvorba layoutů v GIS software,
- export mapových výstupů do PDF, PNG nebo jiných formátů.

Tuto navazující část lze dělat v ArcGIS Pro, QGIS nebo jiném GIS nástroji. Pokud je potřeba software-specific automatizace layoutů, může být přidána jako samostatná optional extension. Není ale součástí hlavního preprocessing workflow.


## 13. Limitace a další kroky

Projekt není plně univerzální ve smyslu „vezmi libovolnou říční síť a vše poběží bez konfigurace“. Stále platí několik poctivých omezení:

- W05 field names (`W05_006`, `W05_009`, `W05_010`) jsou specifické pro původní dataset a musí zůstat v profilu.
- Region assignment používá polygonovou vrstvu prefektur v `regions/`. Kód a název prefektury se hledají podle candidate field names v konfigurační části notebooku.
- Pokud vstup nemá CRS metadata, fallback CRS je kvalifikovaný předpoklad, ne důkaz. Před ostrým použitím je nutný audit extentu a zdrojové dokumentace.
- Délkové metriky závisí na zvoleném metrickém CRS. Notebook používá odhad UTM pro daný rozsah dat, což je praktické pro regionální vrstvy, ale u rozsáhlých nadregionálních sítí může být vhodnější explicitní projekce.
- Source-distance metrika předpokládá orientovaný acyklický graf. Segmenty v cyklech dostávají neurčitelné hodnoty, protože nejdelší cesta v pozitivně váženém cyklu není jednoznačná.

Další rozumný krok je přidat samostatný stylovací předpis pro QGIS/ArcGIS, který už jen čte připravené atributy z exportovaného datasetu.
